In [7]:
# 1. 라이브러리 및 데이터 로드 (순정 550+ 확보용)
import pandas as pd
import numpy as np
import os
import time
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
import lightgbm as lgb

DATA_DIR = r"C:\Users\이호준\OneDrive\바탕 화면\LG aimers\open\data"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "./data" # 코랩 환경 대비

ID_COL = "row_id"
TARGET_COL = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand"] # hand 변수도 범주형 추가

try:
    train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), encoding="utf-8-sig")
    test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), encoding="utf-8-sig", nrows=0).columns
    print("데이터 로드 완료:", train.shape)
except Exception as e:
    print("경로 확인 필요:", e)


데이터 로드 완료: (1475092, 49)


In [8]:
# 2. 피처 설정 및 파이프라인 구축
FEATURES = [c for c in test_cols if c != ID_COL]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

print(f"전체 피처 수: {len(FEATURES)} (범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")

# 주최측과 동일한 안정적인 전처리 (결측치 중앙값 대치 + 라벨 인코딩)
preprocessor = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS),
])

# [핵심] 매우 보수적인 LightGBM 파라미터 (과적합 및 극단적 확률 방지)
# min_child_samples=500 으로 설정하여 한 잎사귀에 데이터가 500개는 모여야 확률을 내도록 강제 겸손화
model = Pipeline([
    ("pre", preprocessor),
    ("clf", lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=6,
        min_child_samples=500,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbosity=-1
    ))
])


전체 피처 수: 47 (범주형 5, 수치형 42)


In [9]:
# 3. 주최측과 동일한 시계열 검증 (미래 참조 누수 방지)
is_val = train["season"] == 2024
X_train, y_train = train.loc[~is_val, FEATURES], train.loc[~is_val, TARGET_COL]
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET_COL]

print(f"학습(2019~2023): {len(X_train)} | 검증(2024): {len(X_val)}")

t = time.time()
model.fit(X_train, y_train)
print(f"학습 완료 :: {time.time() - t:.1f}s")

# 검증 점수 산출
val_pred = model.predict_proba(X_val)[:, 1]

r = y_val.mean()
brier = ((val_pred - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f"Brier Score: {brier:.6f} | 기준선 r(1-r): {baseline_brier:.6f}")
print(f"Validation Score: {score:.2f} (목표: 549.51 이상)")


학습(2019~2023): 1221585 | 검증(2024): 253507
학습 완료 :: 77.8s
Brier Score: 0.248322 | 기준선 r(1-r): 0.249807
Validation Score: 594.60 (목표: 549.51 이상)


In [10]:
# 4. 전체 데이터 재학습 및 모델 저장 (submission.zip 생성)
import zipfile

print("전체 데이터로 최종 모델 학습 중...")
t = time.time()
model.fit(train[FEATURES], train[TARGET_COL])
print(f"재학습 완료 :: {time.time() - t:.1f}s")

os.makedirs("./submission/model", exist_ok=True)
joblib.dump(model, "./submission/model/lgb_pipeline.pkl", compress=3)

# requirements.txt
with open("./submission/requirements.txt", "w") as f:
    f.write("pandas\nscikit-learn\nlightgbm\njoblib\n")

# script.py 작성 (주최측 베이스라인 추론 로직과 100% 동일)
script_code = """import os
import joblib
import pandas as pd

ID_COL = "row_id"
TARGET_COL = "control_success"

def merge_predictions(sub, ids, preds):
    pred_map = dict(zip(ids, preds))
    values, n_missing = [], 0
    for rid, cur in zip(sub[ID_COL], sub[TARGET_COL]):
        p = pred_map.get(rid)
        if p is None:
            n_missing += 1
            values.append(cur)
        else:
            values.append(p)
    sub[TARGET_COL] = values
    return sub

def main():
    TEST_DIR = "./data"
    MODEL_DIR = "./model"
    OUT_DIR = "./output"
    TEST_PATH = os.path.join(TEST_DIR, "test.csv")
    SAMPLE_SUB_PATH = os.path.join(TEST_DIR, "sample_submission.csv")
    MODEL_PATH = os.path.join(MODEL_DIR, "lgb_pipeline.pkl")
    OUT_PATH = os.path.join(OUT_DIR, "submission.csv")

    model = joblib.load(MODEL_PATH)
    
    test = pd.read_csv(TEST_PATH, encoding="utf-8-sig")
    sub = pd.read_csv(SAMPLE_SUB_PATH, encoding="utf-8-sig")
    
    ids = test[ID_COL].tolist()
    X = test.drop(columns=[ID_COL])
    
    preds = model.predict_proba(X)[:, 1] if len(X) else []
    
    sub = merge_predictions(sub, ids, preds)
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    sub.to_csv(OUT_PATH, index=False, encoding="utf-8")

if __name__ == "__main__":
    main()
"""

with open("./submission/script.py", "w", encoding="utf-8") as f:
    f.write(script_code)

# zip 묶기
zip_path = 'baseline_submit.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk('./submission'):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, './submission')
            zipf.write(file_path, arcname)

print("✅ 가장 깨끗한 순정 베이스라인 baseline_submit.zip 생성 완료!")


전체 데이터로 최종 모델 학습 중...
재학습 완료 :: 90.7s
✅ 가장 깨끗한 순정 베이스라인 baseline_submit.zip 생성 완료!
